# E2: Episodic Trace Memory on ALFWorld

Validates the full episodic trace memory pipeline from the RFC spec:

1. **Ingest** expert trajectories as multi-step episodes
2. **Retrieve** analogous episodes by initial-state similarity + reward filter
3. **Inject** retrieved trajectories as procedural context
4. **Evaluate** task completion with and without episodic memory

**Dataset**: AgentInstruct (336 GPT-4 expert ALFWorld trajectories)

| Condition | Description |
|-----------|-------------|
| NoMemory | Zero-shot agent |
| RandomTrajectory | Random past trajectory injected |
| CVX-Episodic | Reward-filtered temporal analogy retrieval |

In [ ]:
import os, json, time
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI

EMBED_MODEL = 'all-MiniLM-L6-v2'
LLM_MODEL = 'gpt-4o-mini'
TOP_K = 3

DATA_DIR = Path('../data/episodic')
DATA_DIR.mkdir(parents=True, exist_ok=True)

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY'
client = OpenAI()
embedder = SentenceTransformer(EMBED_MODEL)
D = embedder.get_sentence_embedding_dimension()
print(f'Embedding: {EMBED_MODEL} (D={D}), LLM: {LLM_MODEL}')

## 1. Load AgentInstruct Trajectories

Expert ALFWorld trajectories generated by GPT-4.

In [ ]:
from datasets import load_dataset

# AgentInstruct: expert ALFWorld trajectories
try:
    ds = load_dataset('zai-org/agentinstruct', split='train')
    print(f'AgentInstruct: {len(ds)} trajectories')
except Exception as e:
    print(f'AgentInstruct not available: {e}')
    print('Generating synthetic ALFWorld-style trajectories for demonstration...')
    
    # Synthetic trajectories for demonstration
    task_types = ['pick', 'clean', 'heat', 'cool', 'examine', 'put']
    objects = ['apple', 'knife', 'mug', 'book', 'pen', 'plate']
    locations = ['kitchen', 'bedroom', 'bathroom', 'living room', 'desk']
    
    synthetic = []
    np.random.seed(42)
    for i in range(200):
        task = np.random.choice(task_types)
        obj = np.random.choice(objects)
        loc = np.random.choice(locations)
        n_steps = np.random.randint(5, 15)
        success = np.random.random() > 0.3  # 70% success rate
        
        steps = []
        for s in range(n_steps):
            action = f'{np.random.choice(["go to", "look at", "take", "use", "put"])} {obj} in {loc}'
            obs = f'You see a {obj} on the {np.random.choice(["table", "shelf", "counter"])} in {loc}.'
            steps.append({'action': action, 'observation': obs})
        
        synthetic.append({
            'task': f'{task} the {obj} in {loc}',
            'steps': steps,
            'reward': 1.0 if success else 0.0,
            'task_type': task,
        })
    
    ds = synthetic
    print(f'Generated {len(ds)} synthetic trajectories')

## 2. Build Episodic Index

In [ ]:
import chronos_vector as cvx

INDEX_PATH = str(DATA_DIR / 'alfworld_episodic.cvx')

def encode_entity(episode_id, step_index):
    return (episode_id << 16) | step_index

if os.path.exists(INDEX_PATH):
    index = cvx.TemporalIndex.load(INDEX_PATH)
    print(f'Loaded: {len(index)} points')
else:
    print('Building episodic memory...')
    index = cvx.TemporalIndex(m=16, ef_construction=100, ef_search=50)
    
    episode_data = {}
    
    for ep_idx, traj in enumerate(ds):
        task = traj['task'] if isinstance(traj, dict) else traj['task']
        steps = traj['steps'] if isinstance(traj, dict) else json.loads(traj['steps'])
        reward = traj['reward'] if isinstance(traj, dict) else float(traj.get('reward', 0))
        
        # Store each step as a temporal vector
        for s_idx, step in enumerate(steps):
            if isinstance(step, dict):
                text = f"Task: {task} | Action: {step.get('action', '')} | Obs: {step.get('observation', '')}"
            else:
                text = f"Task: {task} | Step: {step}"
            
            vec = embedder.encode(text[:500]).tolist()
            eid = encode_entity(ep_idx, s_idx)
            index.insert(entity_id=eid, timestamp=ep_idx * 10000 + s_idx, vector=vec)
        
        episode_data[ep_idx] = {
            'task': task,
            'steps': steps,
            'reward': reward,
            'n_steps': len(steps),
            'task_type': traj.get('task_type', 'unknown'),
        }
    
    index.save(INDEX_PATH)
    with open(str(DATA_DIR / 'alfworld_metadata.json'), 'w') as f:
        json.dump(episode_data, f)
    print(f'Built: {len(index)} points ({len(episode_data)} episodes)')

with open(str(DATA_DIR / 'alfworld_metadata.json')) as f:
    episode_data = {int(k): v for k, v in json.load(f).items()}

n_success = sum(1 for e in episode_data.values() if e['reward'] >= 0.7)
print(f'{len(episode_data)} episodes ({n_success} successful, {len(episode_data)-n_success} failed)')

## 3. Retrieval & Evaluation

In [ ]:
def retrieve_episodes(task_description, top_k=TOP_K, min_reward=0.7):
    """Retrieve successful episodes similar to the given task."""
    query_vec = embedder.encode(f'Task: {task_description}').tolist()
    results = index.search(vector=query_vec, k=top_k * 20)
    
    seen = set()
    episodes = []
    for node_id, ts, score in results:
        ep_idx = node_id // max(1, max(e['n_steps'] for e in episode_data.values()))
        # More robust: use timestamp to find episode
        ep_idx = ts // 10000
        
        if ep_idx in seen or ep_idx not in episode_data:
            continue
        
        ep = episode_data[ep_idx]
        if ep['reward'] < min_reward:  # reward filter!
            continue
        
        seen.add(ep_idx)
        episodes.append({'episode_id': ep_idx, 'similarity': score, **ep})
        
        if len(episodes) >= top_k:
            break
    
    return episodes


def format_trajectory(episodes):
    """Format episodes for LLM prompt."""
    lines = []
    for i, ep in enumerate(episodes, 1):
        outcome = 'Success' if ep['reward'] >= 0.7 else 'Failure'
        lines.append(f'[Past Episode {i} — {outcome}, {ep["n_steps"]} steps]')
        lines.append(f'Task: {ep["task"]}')
        for j, step in enumerate(ep['steps'][:10], 1):  # cap at 10 steps
            if isinstance(step, dict):
                lines.append(f'  Step {j}: {step.get("action", "")} → {step.get("observation", "")[:80]}')
            else:
                lines.append(f'  Step {j}: {str(step)[:100]}')
        lines.append('')
    return '\n'.join(lines)


# Test retrieval
test_task = episode_data[0]['task']
retrieved = retrieve_episodes(test_task)
print(f'Query: {test_task}')
print(f'Retrieved {len(retrieved)} episodes:')
for ep in retrieved:
    print(f'  [{ep["episode_id"]}] sim={ep["similarity"]:.3f}, reward={ep["reward"]}, steps={ep["n_steps"]}: {ep["task"][:50]}')

In [ ]:
def solve_with_llm(task, context=''):
    """Ask LLM to produce a plan for the task."""
    system = (
        'You are an embodied agent in a household. Given a task, produce a step-by-step '
        'action plan. Each step should be one action like "go to kitchen", "take apple", '
        '"put apple in fridge". Output ONLY the numbered steps, no explanation.'
    )
    user_parts = []
    if context:
        user_parts.append('Here are examples of how similar tasks were solved:\n')
        user_parts.append(context)
        user_parts.append('\n---\nNow solve this task:\n')
    user_parts.append(f'Task: {task}')
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': '\n'.join(user_parts)},
        ],
        temperature=0.0,
        max_tokens=512,
    )
    plan = response.choices[0].message.content
    n_steps = len([l for l in plan.strip().split('\n') if l.strip()])
    return plan, n_steps


# Evaluate: compare plan quality across conditions
# Without a real ALFWorld env, we measure:
# 1. Plan specificity (number of concrete action steps)
# 2. Similarity of plan to expert trajectory
# 3. Token efficiency

N_EVAL = min(30, len(episode_data))
eval_episodes = list(episode_data.items())[:N_EVAL]

metrics = {'NoMemory': [], 'RandomTrajectory': [], 'CVX-Episodic': []}

for ep_idx, ep in eval_episodes:
    task = ep['task']
    expert_steps = ep['n_steps']
    
    # NoMemory
    plan, n_steps = solve_with_llm(task)
    metrics['NoMemory'].append({'steps': n_steps, 'expert_steps': expert_steps})
    
    # RandomTrajectory
    random_eps = [episode_data[i] for i in np.random.choice(list(episode_data.keys()), 3)]
    plan, n_steps = solve_with_llm(task, format_trajectory(random_eps))
    metrics['RandomTrajectory'].append({'steps': n_steps, 'expert_steps': expert_steps})
    
    # CVX-Episodic (exclude the current episode)
    retrieved = retrieve_episodes(task, min_reward=0.7)
    retrieved = [e for e in retrieved if e['episode_id'] != ep_idx][:3]
    if retrieved:
        plan, n_steps = solve_with_llm(task, format_trajectory(retrieved))
    else:
        plan, n_steps = solve_with_llm(task)
    metrics['CVX-Episodic'].append({'steps': n_steps, 'expert_steps': expert_steps})
    
    if (ep_idx + 1) % 10 == 0:
        print(f'[{ep_idx+1}/{N_EVAL}] done')

print('\n=== RESULTS ===')
for cond, data in metrics.items():
    mean_steps = np.mean([d['steps'] for d in data])
    mean_expert = np.mean([d['expert_steps'] for d in data])
    step_ratio = mean_steps / max(mean_expert, 1)
    print(f'{cond:20s}: mean steps={mean_steps:.1f} (expert={mean_expert:.1f}, ratio={step_ratio:.2f})')

In [ ]:
import plotly.graph_objects as go

cond_names = list(metrics.keys())
mean_steps = [np.mean([d['steps'] for d in metrics[c]]) for c in cond_names]

fig = go.Figure(data=go.Bar(
    x=cond_names, y=mean_steps,
    text=[f'{s:.1f}' for s in mean_steps],
    textposition='outside',
    marker_color=['#95a5a6', '#3498db', '#e74c3c'],
))
fig.update_layout(
    title=f'Mean Plan Steps by Condition (n={N_EVAL})',
    yaxis_title='Mean Steps', height=400,
)
fig.show()

## Summary

In [ ]:
print('=== E2: EPISODIC ALFWORLD BENCHMARK ===')
print(f'Model: {LLM_MODEL}')
print(f'Episodes: {len(episode_data)} ({n_success} successful)')
print(f'Evaluated: {N_EVAL} tasks')
print()
for cond, data in metrics.items():
    mean_s = np.mean([d['steps'] for d in data])
    print(f'  {cond:20s}: {mean_s:.1f} mean steps')
print()
print('CVX features used: insert, search, episode encoding, reward filtering')
print('Note: Without real ALFWorld env, we compare plan length vs expert.')
print('Full evaluation requires alfworld package + environment execution.')